# Command Line Arguments and User Input in Python

This notebook explores how to handle command line arguments and gather user input in Python applications - essential skills for creating interactive Python scripts and command-line tools.

## Required Libraries

First, let's install the libraries we'll need for this notebook:

In [1]:
# Install required packages
!pip install click pytest-shutil

## 1. Understanding Command Line Arguments

Command line arguments allow users to provide input to a program when it is launched from the command line. This provides a powerful way to make your Python scripts configurable without modifying the code.

Command line arguments are especially useful for:
- Configuring program behavior
- Specifying input/output files
- Setting program options
- Running in different modes (verbose, debug, etc.)

When a Python script is executed, the command line arguments are passed to the script as a list of strings, regardless of their original type. The program is responsible for parsing these strings and converting them to appropriate types.

### Basic Command Line Pattern

A typical command line pattern looks like this:

```
python script.py arg1 arg2 --option1 value1 --flag
```

Where:
- `script.py` is the Python script to run
- `arg1`, `arg2` are positional arguments
- `--option1 value1` is an option with a value
- `--flag` is a flag (boolean option)

## 2. Using the sys Module

The simplest way to access command line arguments is through the `sys` module's `argv` list. This is a list containing the command line arguments passed to the script.

In [2]:
import sys

# Let's create a simple example of how sys.argv works
def show_argv():
    print(f"Program name: {sys.argv[0]}")
    print(f"Number of arguments: {len(sys.argv) - 1}")
    print(f"Arguments: {sys.argv[1:]}")
    
# We can't directly test this in a notebook with real command line args,
# so let's simulate it
original_argv = sys.argv
sys.argv = ["myscript.py", "arg1", "arg2", "--option=value"]
show_argv()
sys.argv = original_argv  # Restore original

Program name: myscript.py
Number of arguments: 3
Arguments: ['arg1', 'arg2', '--option=value']


### Simple Argument Parser using sys.argv

Here's a basic example of a script that processes arguments manually:

In [3]:
def process_args_manually(args):
    if len(args) < 2:
        print("Usage: myscript.py <name> [--greeting=<greeting>]")
        return
    
    name = args[1]
    greeting = "Hello"
    
    # Process optional arguments
    for arg in args[2:]:
        if arg.startswith("--greeting="):
            greeting = arg.split("=")[1]
    
    print(f"{greeting}, {name}!")

# Test with example arguments
test_args = ["myscript.py", "World", "--greeting=Hi"]
process_args_manually(test_args)

test_args = ["myscript.py", "Python"]
process_args_manually(test_args)

Hi, World!
Hello, Python!


### Limitations of sys.argv

While `sys.argv` provides direct access to command line arguments, it has some limitations:

1. Manual parsing can be error-prone
2. No built-in validation or type conversion
3. No automatic help generation
4. Handling complex argument patterns becomes tedious

This is why Python provides better libraries for command line argument parsing.

## 3. Using argparse Module

The `argparse` module provides a more sophisticated command-line argument parsing library. It automatically generates help and usage messages and handles type conversion and validation.

In [4]:
import argparse

def parse_with_argparse(args=None):
    parser = argparse.ArgumentParser(description="A simple example of argparse usage.")
    
    # Add a positional argument
    parser.add_argument("name", help="Name to greet")
    
    # Add an optional argument with a default value
    parser.add_argument("--greeting", default="Hello", help="Greeting to use")
    
    # Add a flag (boolean option)
    parser.add_argument("--uppercase", action="store_true", help="Convert the greeting to uppercase")
    
    # Add an argument with type conversion
    parser.add_argument("--repeat", type=int, default=1, help="Number of times to repeat the greeting")
    
    # Parse arguments
    args = parser.parse_args(args)
    
    # Process arguments
    greeting = args.greeting
    if args.uppercase:
        greeting = greeting.upper()
    
    for _ in range(args.repeat):
        print(f"{greeting}, {args.name}!")
    
    return args

# Test with example arguments
print("Example 1: Basic usage")
args1 = parse_with_argparse(["World"])
print("\nExample 2: With options")
args2 = parse_with_argparse(["Python", "--greeting", "Hi", "--repeat", "3"])
print("\nExample 3: With uppercase flag")
args3 = parse_with_argparse(["Developer", "--greeting", "Welcome", "--uppercase"])

Example 1: Basic usage
Hello, World!

Example 2: With options
Hi, Python!
Hi, Python!
Hi, Python!

Example 3: With uppercase flag
WELCOME, Developer!


### Advanced argparse Features

The `argparse` module provides many advanced features for complex command line interfaces:

In [5]:
def advanced_argparse_example(args=None):
    parser = argparse.ArgumentParser(description="Demonstrate advanced argparse features")
    
    # Subcommands
    subparsers = parser.add_subparsers(dest="command", help="Commands")
    
    # Create parsers for each command
    create_parser = subparsers.add_parser("create", help="Create a new item")
    create_parser.add_argument("name", help="Name of the item to create")
    
    delete_parser = subparsers.add_parser("delete", help="Delete an existing item")
    delete_parser.add_argument("id", type=int, help="ID of the item to delete")
    
    # Mutually exclusive group
    group = parser.add_mutually_exclusive_group()
    group.add_argument("-v", "--verbose", action="store_true", help="Increase verbosity")
    group.add_argument("-q", "--quiet", action="store_true", help="Decrease verbosity")
    
    # Parse arguments
    parsed_args = parser.parse_args(args)
    
    # Process based on command
    if parsed_args.command == "create":
        print(f"Creating item: {parsed_args.name}")
    elif parsed_args.command == "delete":
        print(f"Deleting item with ID: {parsed_args.id}")
    
    # Handle verbosity
    if hasattr(parsed_args, 'verbose') and parsed_args.verbose:
        print("Verbose mode enabled")
    elif hasattr(parsed_args, 'quiet') and parsed_args.quiet:
        print("Quiet mode enabled")
    
    return parsed_args

# Test with example arguments
print("Example 1: Create command")
args1 = advanced_argparse_example(["create", "new_item", "-v"])
print("\nExample 2: Delete command")
args2 = advanced_argparse_example(["delete", "42", "-q"])

Example 1: Create command


usage: ipykernel_launcher.py [-h] [-v | -q] {create,delete} ...
ipykernel_launcher.py: error: unrecognized arguments: -v


SystemExit: 2

C:\Users\pavel\projects\ai-ml\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3675: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 4. Input from the User

Command line arguments are provided when the script starts, but sometimes you need to get input from the user during execution. Python provides several ways to do this.

### Basic Input Function

The simplest way to get input from a user is with the `input()` function:

In [6]:
def basic_input_example():
    # In a notebook, we'll just simulate this with predefined inputs
    # In a real script, you'd use:
    # name = input("Enter your name: ")
    
    print("Prompt: Enter your name: ")
    name = "John"  # Simulating user input
    print(f"User entered: {name}")
    
    print("\nPrompt: Enter your age: ")
    age_input = "30"  # Simulating user input
    print(f"User entered: {age_input}")
    
    # Converting input to the appropriate type
    try:
        age = int(age_input)
        print(f"Hello {name}, you are {age} years old.")
    except ValueError:
        print("That's not a valid age!")

basic_input_example()

Prompt: Enter your name: 
User entered: John

Prompt: Enter your age: 
User entered: 30
Hello John, you are 30 years old.


### Input Validation

When getting input from users, it's important to validate it to ensure it's in the expected format:

In [7]:
def get_valid_age(prompt_msg="Enter your age: "):
    # Simulation of user inputs for demonstration
    test_inputs = ["twenty", "-5", "0", "120", "42"]
    
    for test_input in test_inputs:
        print(f"Prompt: {prompt_msg}")
        print(f"User entered: {test_input}")
        
        try:
            age = int(test_input)
            if 0 < age < 120:  # Basic validation
                print(f"Valid age entered: {age}\n")
                return age
            else:
                print("Age must be between 1 and 119\n")
        except ValueError:
            print("Please enter a valid number\n")
    
    print("Maximum attempts reached. Using default age of 30")
    return 30

# Get a valid age
valid_age = get_valid_age()

Prompt: Enter your age: 
User entered: twenty
Please enter a valid number

Prompt: Enter your age: 
User entered: -5
Age must be between 1 and 119

Prompt: Enter your age: 
User entered: 0
Age must be between 1 and 119

Prompt: Enter your age: 
User entered: 120
Age must be between 1 and 119

Prompt: Enter your age: 
User entered: 42
Valid age entered: 42



### Password Input

For sensitive information like passwords, you might want to hide the input as it's being typed. The `getpass` module provides this functionality:

In [8]:
import getpass

def password_input_example():
    # In a real script, you'd use:
    # username = input("Username: ")
    # password = getpass.getpass("Password: ")
    
    print("Prompt: Username: ")
    username = "admin"  # Simulating user input
    print(f"User entered: {username}")
    
    print("\nPrompt: Password: ")
    password = "secure_password"  # Simulating password input (would be hidden in real use)
    print("User entered: *********")
    
    # Simple validation (in real applications, you'd compare to stored, hashed passwords)
    if username == "admin" and password == "secure_password":
        print("Login successful")
    else:
        print("Invalid credentials")

password_input_example()

Prompt: Username: 
User entered: admin

Prompt: Password: 
User entered: *********
Login successful


## 5. Working with Files as Input

Command-line tools often need to process files. Let's look at how to handle file input from command line arguments.

In [9]:
import os
import tempfile

def file_processor(input_file, output_file=None):
    """Process an input file and optionally write results to output file"""
    try:
        # Read input file
        with open(input_file, 'r') as f:
            content = f.read()
            print(f"Read {len(content)} characters from {input_file}")
            
            # Process content (simple example: count words)
            word_count = len(content.split())
            result = f"Word count: {word_count}\n"
            print(result.strip())
            
            # Write to output file if specified
            if output_file:
                with open(output_file, 'w') as out_f:
                    out_f.write(result)
                print(f"Results written to {output_file}")
                
    except FileNotFoundError:
        print(f"Error: File '{input_file}' not found")
    except IOError as e:
        print(f"I/O error: {e}")

# Create a temporary file for testing
with tempfile.NamedTemporaryFile(delete=False, mode='w', suffix='.txt') as tmp:
    tmp.write("This is a sample file.\nIt contains some text for our file processing example.\nWe will count words in this file.")
    temp_filename = tmp.name

# Create a temporary output file
output_filename = tempfile.mktemp(suffix='.txt')

# Process the file
print("Example: Processing a file")
file_processor(temp_filename, output_filename)

# Clean up temporary files
os.remove(temp_filename)
if os.path.exists(output_filename):
    os.remove(output_filename)

Example: Processing a file
Read 111 characters from C:\Users\pavel\AppData\Local\Temp\tmp92qbonau.txt
Word count: 21
Results written to C:\Users\pavel\AppData\Local\Temp\tmppmfm4gye.txt


### Creating a File Processing Script with argparse

Let's combine what we've learned to create a complete file processing script:

In [10]:
def file_processing_script(args=None):
    parser = argparse.ArgumentParser(description="Process a text file and produce statistics.")
    parser.add_argument("input", help="Input file path")
    parser.add_argument("-o", "--output", help="Output file path")
    parser.add_argument("-c", "--count", choices=['chars', 'words', 'lines'], default='words',
                        help="What to count (default: words)")
    parser.add_argument("-v", "--verbose", action="store_true", help="Enable verbose output")
    
    parsed_args = parser.parse_args(args)
    
    try:
        # Check if input file exists
        if not os.path.exists(parsed_args.input):
            raise FileNotFoundError(f"Input file '{parsed_args.input}' not found")
        
        # Read the file
        with open(parsed_args.input, 'r') as f:
            content = f.read()
            
        if parsed_args.verbose:
            print(f"Read {len(content)} characters from {parsed_args.input}")
        
        # Process based on count type
        if parsed_args.count == 'chars':
            result = len(content)
            result_type = "characters"
        elif parsed_args.count == 'words':
            result = len(content.split())
            result_type = "words"
        else:  # lines
            result = len(content.splitlines())
            result_type = "lines"
        
        # Print result
        output_text = f"The file contains {result} {result_type}.\n"
        print(output_text.strip())
        
        # Write to output file if specified
        if parsed_args.output:
            with open(parsed_args.output, 'w') as f:
                f.write(output_text)
            if parsed_args.verbose:
                print(f"Results written to {parsed_args.output}")
                
    except FileNotFoundError as e:
        print(f"Error: {e}")
    except IOError as e:
        print(f"I/O error: {e}")
    
    return parsed_args

# Create a temporary file for testing
with tempfile.NamedTemporaryFile(delete=False, mode='w', suffix='.txt') as tmp:
    tmp.write("This is a sample file.\nIt contains some text for our file processing example.\nWe will count words in this file.")
    temp_filename = tmp.name

# Create a temporary output file
output_filename = tempfile.mktemp(suffix='.txt')

# Test the script with different options
print("Example 1: Count words (default)")
args1 = file_processing_script([temp_filename, "--verbose"])

print("\nExample 2: Count characters")
args2 = file_processing_script([temp_filename, "-c", "chars"])

print("\nExample 3: Count lines with output file")
args3 = file_processing_script([temp_filename, "-c", "lines", "-o", output_filename, "-v"])

# Clean up temporary files
os.remove(temp_filename)
if os.path.exists(output_filename):
    os.remove(output_filename)

Example 1: Count words (default)
Read 111 characters from C:\Users\pavel\AppData\Local\Temp\tmp5_m7qwb0.txt
The file contains 21 words.

Example 2: Count characters
The file contains 111 characters.

Example 3: Count lines with output file
Read 111 characters from C:\Users\pavel\AppData\Local\Temp\tmp5_m7qwb0.txt
The file contains 3 lines.
Results written to C:\Users\pavel\AppData\Local\Temp\tmpbgpm5mr6.txt


## 6. Command Line Arguments with Click

While `argparse` is powerful, the `click` library provides an even more elegant way to create command-line interfaces. It uses decorators to define commands and arguments, making the code more readable.

In [11]:
import click
from click.testing import CliRunner

@click.command()
@click.argument('name')
@click.option('--greeting', '-g', default='Hello', help='Greeting to use')
@click.option('--repeat', '-r', default=1, type=int, help='Number of times to repeat')
@click.option('--uppercase/--no-uppercase', default=False, help='Convert greeting to uppercase')
def greet(name, greeting, repeat, uppercase):
    """Simple greeting program."""
    if uppercase:
        greeting = greeting.upper()
    for _ in range(repeat):
        click.echo(f"{greeting}, {name}!")

# Test the click command
runner = CliRunner()

print("Example 1: Basic usage")
result = runner.invoke(greet, ['World'])
print(result.output)

print("Example 2: With options")
result = runner.invoke(greet, ['Alice', '--greeting', 'Hi', '--repeat', '3'])
print(result.output)

print("Example 3: With uppercase")
result = runner.invoke(greet, ['Bob', '--uppercase'])
print(result.output)

Example 1: Basic usage
Hello, World!

Example 2: With options
Hi, Alice!
Hi, Alice!
Hi, Alice!

Example 3: With uppercase
HELLO, Bob!



### Advanced Click Features

Click offers many advanced features like command groups, file handling, and progress bars:

In [12]:
@click.group()
def cli():
    """File utility commands."""
    pass

@cli.command()
@click.argument('input_file', type=click.Path(exists=True))
@click.option('--count', '-c', type=click.Choice(['chars', 'words', 'lines']), 
              default='words', help='What to count')
def count(input_file, count):
    """Count elements in a file."""
    with open(input_file, 'r') as f:
        content = f.read()
    
    if count == 'chars':
        result = len(content)
        result_type = "characters"
    elif count == 'words':
        result = len(content.split())
        result_type = "words"
    else:  # lines
        result = len(content.splitlines())
        result_type = "lines"
        
    click.echo(f"The file contains {result} {result_type}.")

@cli.command()
@click.argument('input_file', type=click.Path(exists=True))
@click.argument('output_file', type=click.Path())
@click.option('--uppercase/--no-uppercase', default=False, help='Convert to uppercase')
def convert(input_file, output_file, uppercase):
    """Convert file contents."""
    with open(input_file, 'r') as f_in:
        content = f_in.read()
    
    if uppercase:
        content = content.upper()
    
    with open(output_file, 'w') as f_out:
        f_out.write(content)
        
    click.echo(f"File converted and saved to {output_file}")

# Test the click group commands
runner = CliRunner()

# Create a temporary file for testing
with tempfile.NamedTemporaryFile(delete=False, mode='w', suffix='.txt') as tmp:
    tmp.write("This is a sample file.\nIt contains some text for our file processing example.\nWe will count words in this file.")
    temp_filename = tmp.name

# Create a temporary output file
output_filename = tempfile.mktemp(suffix='.txt')

print("Example 1: Count command")
result = runner.invoke(cli, ['count', temp_filename, '-c', 'words'])
print(result.output)

print("\nExample 2: Convert command")
result = runner.invoke(cli, ['convert', temp_filename, output_filename, '--uppercase'])
print(result.output)

# Clean up temporary files
os.remove(temp_filename)
if os.path.exists(output_filename):
    os.remove(output_filename)

Example 1: Count command
The file contains 21 words.


Example 2: Convert command
File converted and saved to C:\Users\pavel\AppData\Local\Temp\tmpsd_bjjv5.txt



## 7. Error Handling with Command Line Inputs

Proper error handling is essential for command line applications to provide a good user experience.

In [13]:
def robust_file_processing(args=None):
    parser = argparse.ArgumentParser(description="Process a text file with robust error handling.")
    parser.add_argument("input", help="Input file path")
    parser.add_argument("-o", "--output", help="Output file path")
    parser.add_argument("-n", "--number", type=int, help="A number parameter for demonstration")
    
    try:
        parsed_args = parser.parse_args(args)
        
        # Input file validation
        if not os.path.exists(parsed_args.input):
            raise FileNotFoundError(f"Input file not found: {parsed_args.input}")
        
        if not os.path.isfile(parsed_args.input):
            raise IsADirectoryError(f"Expected a file, got a directory: {parsed_args.input}")
        
        # Output file validation
        if parsed_args.output:
            output_dir = os.path.dirname(parsed_args.output) or '.'
            if output_dir != '.' and not os.path.exists(output_dir):
                raise NotADirectoryError(f"Output directory does not exist: {output_dir}")
            
            if os.path.exists(parsed_args.output) and not os.access(parsed_args.output, os.W_OK):
                raise PermissionError(f"No write permission for output file: {parsed_args.output}")
        
        # Number parameter validation
        if parsed_args.number is not None and parsed_args.number < 0:
            raise ValueError(f"Number must be non-negative: {parsed_args.number}")
        
        # If all validations pass, process the file
        with open(parsed_args.input, 'r') as f:
            content = f.read()
            word_count = len(content.split())
            result = f"The file contains {word_count} words.\n"
            print(result.strip())
        
        # Write to output if specified
        if parsed_args.output:
            with open(parsed_args.output, 'w') as f:
                f.write(result)
            print(f"Result written to {parsed_args.output}")
        
        return 0  # Success
        
    except FileNotFoundError as e:
        print(f"Error: {e}")
        return 1  # File not found error
    except IsADirectoryError as e:
        print(f"Error: {e}")
        return 2  # Expected file, got directory
    except NotADirectoryError as e:
        print(f"Error: {e}")
        return 3  # Output directory does not exist
    except PermissionError as e:
        print(f"Error: {e}")
        return 4  # Permission error
    except ValueError as e:
        print(f"Error: {e}")
        return 5  # Value error
    except Exception as e:
        print(f"Unexpected error: {e}")
        return 99  # Unknown error

# Create a temporary file for testing
with tempfile.NamedTemporaryFile(delete=False, mode='w', suffix='.txt') as tmp:
    tmp.write("This is a sample file.\nIt contains some text for our file processing example.")
    temp_filename = tmp.name

# Create a temporary output file
output_filename = tempfile.mktemp(suffix='.txt')

# Test with valid arguments
print("Example 1: Valid arguments")
exit_code = robust_file_processing([temp_filename, "-o", output_filename, "-n", "5"])
print(f"Exit code: {exit_code}\n")

# Test with non-existent input file
print("Example 2: Non-existent input file")
exit_code = robust_file_processing(["non_existent_file.txt"])
print(f"Exit code: {exit_code}\n")

# Test with invalid number
print("Example 3: Invalid number")
exit_code = robust_file_processing([temp_filename, "-n", "-5"])
print(f"Exit code: {exit_code}")

# Clean up temporary files
os.remove(temp_filename)
if os.path.exists(output_filename):
    os.remove(output_filename)

Example 1: Valid arguments
The file contains 14 words.
Result written to C:\Users\pavel\AppData\Local\Temp\tmpxbab0k6k.txt
Exit code: 0

Example 2: Non-existent input file
Error: Input file not found: non_existent_file.txt
Exit code: 1

Example 3: Invalid number
Error: Number must be non-negative: -5
Exit code: 5


### Graceful Exits

Using appropriate exit codes can help in scripting and automation:

In [14]:
import sys

def graceful_exit_example(args=None):
    parser = argparse.ArgumentParser(description="Demonstrate graceful exits")
    parser.add_argument("action", choices=["success", "warning", "error"], 
                       help="Action to simulate")
    
    parsed_args = parser.parse_args(args)
    
    try:
        if parsed_args.action == "success":
            print("Operation completed successfully.")
            return_code = 0
        elif parsed_args.action == "warning":
            print("Operation completed with warnings.")
            return_code = 1
        else:  # error
            print("Operation failed.")
            return_code = 2
            
        # In a real script, you would use sys.exit(return_code)
        # But for our notebook example, we'll just return it
        print(f"Would exit with code {return_code}")
        return return_code
        
    except KeyboardInterrupt:
        print("\nOperation cancelled by user.")
        return 130  # Standard code for script terminated by Control-C
    except Exception as e:
        print(f"Unexpected error: {e}")
        return 1  # General error

# Test with different actions
print("Example 1: Success action")
return_code = graceful_exit_example(["success"])
print(f"Returned: {return_code}\n")

print("Example 2: Warning action")
return_code = graceful_exit_example(["warning"])
print(f"Returned: {return_code}\n")

print("Example 3: Error action")
return_code = graceful_exit_example(["error"])
print(f"Returned: {return_code}")

Example 1: Success action
Operation completed successfully.
Would exit with code 0
Returned: 0

Example 2: Warning action
Operation completed with warnings.
Would exit with code 1
Returned: 1

Example 3: Error action
Operation failed.
Would exit with code 2
Returned: 2


## 8. Practical Examples

Let's bring together what we've learned with some practical examples of command-line tools.

### Example 1: Simple Calculator

In [15]:
def calculator(args=None):
    parser = argparse.ArgumentParser(description="Simple command-line calculator.")
    parser.add_argument("expression", help="Mathematical expression to evaluate")
    
    parsed_args = parser.parse_args(args)
    
    try:
        # CAUTION: eval is unsafe for user input in production code
        # This is just for demonstration purposes
        result = eval(parsed_args.expression)
        print(f"{parsed_args.expression} = {result}")
        return 0
    except SyntaxError:
        print(f"Error: Invalid expression syntax: {parsed_args.expression}")
        return 1
    except Exception as e:
        print(f"Error: {e}")
        return 1

# Test with various expressions
print("Example 1: Simple addition")
calculator(["2+2"])

print("\nExample 2: More complex expression")
calculator(["(10*5)/2 + 3**2"])

print("\nExample 3: Invalid expression")
calculator(["2++2"])

Example 1: Simple addition
2+2 = 4

Example 2: More complex expression
(10*5)/2 + 3**2 = 34.0

Example 3: Invalid expression
2++2 = 4


0

### Example 2: Task List Manager (Using Click)

In [16]:
import json

@click.group()
def tasks():
    """Simple task management CLI."""
    pass

# Helper function to load and save tasks
def get_tasks_file():
    # For testing, create a temporary file
    return tempfile.mktemp(suffix='.json')

def load_tasks(file_path):
    if not os.path.exists(file_path):
        return []
    try:
        with open(file_path, 'r') as f:
            return json.load(f)
    except json.JSONDecodeError:
        return []

def save_tasks(tasks, file_path):
    with open(file_path, 'w') as f:
        json.dump(tasks, f)

@tasks.command()
@click.argument('description')
def add(description):
    """Add a new task."""
    file_path = get_tasks_file()
    task_list = load_tasks(file_path)
    
    # Create a new task
    task = {
        "id": len(task_list) + 1,
        "description": description,
        "done": False
    }
    
    task_list.append(task)
    save_tasks(task_list, file_path)
    click.echo(f"Task added: {description}")

@tasks.command()
def list():
    """List all tasks."""
    file_path = get_tasks_file()
    task_list = load_tasks(file_path)
    
    if not task_list:
        click.echo("No tasks found.")
        return
    
    click.echo("ID  Done  Description")
    click.echo("--  ----  -----------")
    for task in task_list:
        done_mark = "[x]" if task["done"] else "[ ]"
        click.echo(f"{task['id']:2}  {done_mark}  {task['description']}")

@tasks.command()
@click.argument('task_id', type=int)
def done(task_id):
    """Mark a task as done."""
    file_path = get_tasks_file()
    task_list = load_tasks(file_path)
    
    for task in task_list:
        if task["id"] == task_id:
            task["done"] = True
            save_tasks(task_list, file_path)
            click.echo(f"Task {task_id} marked as done.")
            return
    
    click.echo(f"Error: Task {task_id} not found.")

@tasks.command()
@click.argument('task_id', type=int)
def remove(task_id):
    """Remove a task."""
    file_path = get_tasks_file()
    task_list = load_tasks(file_path)
    
    for i, task in enumerate(task_list):
        if task["id"] == task_id:
            del task_list[i]
            save_tasks(task_list, file_path)
            click.echo(f"Task {task_id} removed.")
            return
    
    click.echo(f"Error: Task {task_id} not found.")

# Test the task manager
runner = CliRunner()

print("Example: Task Manager Demo")

print("\n1. Adding tasks")
result = runner.invoke(tasks, ['add', 'Buy groceries'])
print(result.output)
result = runner.invoke(tasks, ['add', 'Finish homework'])
print(result.output)
result = runner.invoke(tasks, ['add', 'Call mom'])
print(result.output)

print("\n2. Listing tasks")
result = runner.invoke(tasks, ['list'])
print(result.output)

print("\n3. Marking a task as done")
result = runner.invoke(tasks, ['done', '2'])
print(result.output)

print("\n4. Listing tasks again")
result = runner.invoke(tasks, ['list'])
print(result.output)

print("\n5. Removing a task")
result = runner.invoke(tasks, ['remove', '1'])
print(result.output)

print("\n6. Final task list")
result = runner.invoke(tasks, ['list'])
print(result.output)

Example: Task Manager Demo

1. Adding tasks
Task added: Buy groceries

Task added: Finish homework

Task added: Call mom


2. Listing tasks
No tasks found.


3. Marking a task as done
Error: Task 2 not found.


4. Listing tasks again
No tasks found.


5. Removing a task
Error: Task 1 not found.


6. Final task list
No tasks found.



## Summary

In this notebook, we've explored how to work with command line arguments and user input in Python:

1. **Understanding Command Line Arguments**: We learned about the basic patterns and uses of command line arguments.

2. **Using the sys Module**: We saw how to access raw command line arguments with `sys.argv`.

3. **Using argparse Module**: We explored the standard library's powerful argument parsing capabilities.

4. **Input from the User**: We learned how to get and validate user input during program execution.

5. **Working with Files as Input**: We saw how to process files specified as command line arguments.

6. **Command Line Arguments with Click**: We explored a more elegant, decorator-based approach to CLI creation.

7. **Error Handling with Command Line Inputs**: We learned best practices for robust error handling in CLI applications.

8. **Practical Examples**: We brought everything together with real-world examples.

Command line arguments and user input are fundamental to creating flexible Python applications that can be used in scripts, automation, and interactive contexts. With these tools and techniques, you can build professional-grade command-line interfaces for your Python programs.

## Additional Resources

- [Official Python argparse documentation](https://docs.python.org/3/library/argparse.html)
- [Click documentation](https://click.palletsprojects.com/)
- [Python input() function documentation](https://docs.python.org/3/library/functions.html#input)
- [PEP 8 - Style Guide for Python Code](https://www.python.org/dev/peps/pep-0008/)
- [Python Command Line Arguments - Real Python tutorial](https://realpython.com/python-command-line-arguments/)